In [1]:
import sys
from pathlib import Path
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))
print(BASE_DIR)

/media/ashfaque/datas/ML-projects/squad_optimization


In [2]:
import json

import numpy as np
import pandas as pd

from src.esp_solver import SquadOptimizatoinProblem, MicroRoleAwareDuplicateRepair
# from src.formations import FORMATIONS_SPEC, MICRO_TO_SUBROLE, GLOBAL_ROLE_MAPING

from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.core.problem import ElementwiseProblem, Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.pm import PolynomialMutation

# ESP algorithm with NSGA II
1. implementing with one optimization function, maximizing the overall score of the players and see the pareto front solutions.

In [3]:
player_data_file = BASE_DIR /"data"/"final_squad_cleaned.json"

with open(player_data_file, "r") as f:
    player_squad = json.load(f) 

budget = 1_000_000

#define the problem
problem = SquadOptimizatoinProblem(player_squad,budget=budget)

# configure the algorithm
algorithm = NSGA2(pop_size=10,
                    sampling= IntegerRandomSampling(),
                    crossover=TwoPointCrossover(prob=0.9),
                    mutation =PolynomialMutation(eta=30,prob=1/11),
                    repair = MicroRoleAwareDuplicateRepair(),
                    eliminate_duplicates=True)

# run optimization
result = minimize( problem,
                    algorithm,
                    ('n_gen', 20),
                    seed=1,
                    verbose=True
                        )

n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |       10 |      1 |  0.000000E+00 |  0.000000E+00 |             - |             -
     2 |       20 |      1 |  0.000000E+00 |  0.000000E+00 |  9.0000000000 |         ideal
     3 |       30 |      1 |  0.000000E+00 |  0.000000E+00 |  0.000000E+00 |             f
     4 |       40 |      1 |  0.000000E+00 |  0.000000E+00 |  5.0000000000 |         ideal
     5 |       50 |      1 |  0.000000E+00 |  0.000000E+00 |  1.000000E+01 |         ideal
     6 |       60 |      1 |  0.000000E+00 |  0.000000E+00 |  2.0000000000 |         ideal
     7 |       70 |      1 |  0.000000E+00 |  0.000000E+00 |  0.000000E+00 |             f
     8 |       80 |      1 |  0.000000E+00 |  0.000000E+00 |  3.0000000000 |         ideal
     9 |       90 |      1 |  0.000000E+00 |  0.000000E+00 |  0.000000E+00 |             f
    10 |      100 |      1 |  0.000000E+00 |  0.000000E+00 |  2.0000000000 |         ideal

In [ ]:
if result.X.ndim == 1:
    result.X = result.X[np.newaxis, :]

# print('length of results ', result.X)
# # 4. Extract results
print("Number of solutions:", len(result.F))
print("\nBest teams (Pareto front):")

for i in range(len(result.X)):
    genome = result.X[i]
    # quit()
    objectives = result.F[i]
    # print(genome)
    unique = set(genome[1:])
    if len(unique)<10:      
        print(f'solution {i} contains duplicate players')

# Decode genome to actual team
idx=1
gk = problem.players_by_pos['GK'][int(genome[0])]
outfield_players = []
outfield_players_role = []

roles_genome = problem.assigned_roles[tuple(genome)]

# print(problem.assigned_roles[tuple(genome)])
# quit()
print(roles_genome)
print(genome)

for pos in ['DF','MF','FW']:
    for _ in range(problem.formation[pos]):
        outfield_players.append(problem.players_by_pos[pos][int(genome[idx])])
        try:
            outfield_players_role.append(roles_genome[idx])
        except:
            print(idx,genome[idx],roles_genome)
            quit()
        idx+=1

team = [gk] + outfield_players
team_role = ['GK']+outfield_players_role
# team selected        
print(f"\nTeam {i+1}:")
# print(f" Total rating: {-objectives[0]:.1f}")
# print(f"  Overall: {-objectives[1]:.1f}")
# # print(f"  Potential: {-objectives[2]:.1f}")
# print(f"  Attack: {-objectives[2]:.1f}")
# print(f"  Defense: {-objectives[3]:.1f}")
# print(f"  GK: {-objectives[4]:.1f}")
# print(f"  Players: {[p['Name'] for p in team]}")
rows = []
print(genome)
for idx,p in enumerate(team):
    rows.append({
        "Name": p["Name"],
        "Position": p["GlobalPos"],
        "Overall": p["Overall"],
        'role':team_role[idx],
        'Wage' : p['WageEUR']
    })

df = pd.DataFrame(rows)
print(df)
print(f"Total cost of the team: ",sum(df['Wage']))

Number of solutions: 1

Best teams (Pareto front):
['GK', np.str_('CB'), np.str_('CB'), np.str_('LB'), np.str_('RWB'), np.str_('CM'), np.str_('CDM'), np.str_('RM'), np.str_('LW'), np.str_('RW'), np.str_('ST')]
[ 129  102  976  743  154 1083  214 1333  313  560   93]

Team 1:
[ 129  102  976  743  154 1083  214 1333  313  560   93]
              Name                                          Position  Overall  \
0    P. Bernardoni                                      {'GK': 'GK'}       76   
1       G. Mancini                                      {'CB': 'DF'}       81   
2    Hugo Goiteira                                      {'CB': 'DF'}       73   
3        N. Schulz                          {'LB': 'DF', 'LM': 'MF'}       74   
4   L. Klostermann             {'RWB': 'DF', 'CB': 'DF', 'RB': 'DF'}       80   
5         O. Tufan                         {'CM': 'MF', 'CAM': 'MF'}       74   
6            Allan                         {'CDM': 'MF', 'CM': 'MF'}       80   
7        J. Sambia 

In [ ]:
from src.create_pitch import plot_team

